# 02b — Run the Intake Agent

Inference layer: runs legal intake submissions through the agent (built from `agent_lib.py`), persists structured intake profiles to the Delta-backed intake state store, and populates the HITL attorney-review queue.

### Key Functions

- Process incoming legal intake narratives through the ReAct agent
- Execute semantic retrieval, conflict checking, and case routing workflows
- Persist structured intake profiles to the Delta-backed state store
- Queue completed intakes for attorney review (`reviewed = false`)
- Demonstrate the system's three core behaviors:
  - Case routing
  - Conflict checking
  - Graceful out-of-scope rejection

### Notes
**Switch this notebook to Serverless CPU compute to run all. This Python notebook is attached to a SQL warehouse, which only supports SQL and Markdown cells.

This notebook preserves the original `provision_text` widget and `run_react_agent()` function signature to maintain compatibility with downstream evaluation and benchmarking workflows.

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("llm_endpoint", "anthropic-claude-sonnet-4-6", "LLM Serving Endpoint")

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack — pinned for reproducibility
# Uninstall to ensure clean state
%pip uninstall -y langgraph langgraph-checkpoint langgraph-prebuilt langgraph-sdk
# Install fresh
%pip install --upgrade langgraph databricks-langchain databricks-vectorsearch==0.75

Found existing installation: langgraph 1.0.10
Uninstalling langgraph-1.0.10:
  Successfully uninstalled langgraph-1.0.10
Found existing installation: langgraph-checkpoint 4.1.1
Uninstalling langgraph-checkpoint-4.1.1:
  Successfully uninstalled langgraph-checkpoint-4.1.1
Found existing installation: langgraph-prebuilt 1.0.13
Uninstalling langgraph-prebuilt-1.0.13:
  Successfully uninstalled langgraph-prebuilt-1.0.13
Found existing installation: langgraph-sdk 0.3.15
Uninstalling langgraph-sdk-0.3.15:
  Successfully uninstalled langgraph-sdk-0.3.15
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Using cached langgraph-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached langgraph-1.0.10-py3-

In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import Libraries, Configure Agent, and Setup MLflow
import sys, os, json
import importlib
sys.path.append(os.getcwd())
 
import mlflow
import agent_lib
importlib.reload(agent_lib)  # Reload to pick up any file changes
from pyspark.sql import functions as F
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()
 
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
INTAKE_TABLE = f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.lexpath_intake_profiles"
 
executor = agent_lib.build_agent(LLM_ENDPOINT)
print(f"Agent ready on {LLM_ENDPOINT}")

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels, 10 conflict matters
Agent ready on anthropic-claude-sonnet-4-6


In [0]:
# Inference layer + Delta-backed intake state store

def save_intake_profile(raw_text: str, profile: dict) -> None:
    """Persist the structured intake profile for HITL attorney review."""
    row = spark.createDataFrame(
        [(raw_text, json.dumps(profile), profile.get("status", ""),
          profile.get("predicted_category", ""), profile.get("practice_area", ""),
          profile.get("conflict_status", ""))],
        schema=("raw_intake_text string, profile_json string, status string, "
                "predicted_category string, practice_area string, conflict_status string"),
    ).withColumn("created_at", F.current_timestamp()).withColumn("reviewed", F.lit(False))
    row.write.format("delta").mode("append").saveAsTable(INTAKE_TABLE)
 
def run_react_agent(text_payload: str) -> str:
    """
    Core ReAct Agent processing logic. 
    Runs the full tool loop, persists the intake profile, and returns the
    predicted LEDGAR category label for downstream evaluation.
    """
    if not text_payload.strip():
        return "Unknown"
 
    result = executor.invoke({"input": text_payload})
    profile = agent_lib.extract_json(result.get("output", ""))
    if not profile:
        return "Unknown"
 
    save_intake_profile(text_payload, profile)
    return profile.get("predicted_category") or profile.get("status", "Unknown")

In [0]:
# Standalone widget-driven test
input_text = dbutils.widgets.get("provision_text")
if input_text:
    route = run_react_agent(input_text)
    print(f"🤖 Agent Predicted Route: {route}")

In [0]:
# Demo scenarios: normal, conflict, out-of-scope
demo_intakes = [
    # 1. Normal intake — should retrieve, classify, route; no parties → NOT_RUN
    "My business partner and I signed an agreement that says disputes go to arbitration, "
    "but now they filed a lawsuit in court instead. I want to enforce the arbitration clause.",
 
    # 2. Conflict scenario — names an existing client from lexpath_conflicts (01c)
    "I want to sue Atlas Manufacturing. I was injured by one of their forklifts at a "
    "warehouse in March and they refuse to cover my medical bills. My name is Paul Vance.",
 
    # 3. Out-of-scope — should be gracefully rejected without tool calls
    "Can you just tell me whether I'd win if I represented myself? Give me your legal "
    "opinion on my chances.",
]
 
for i, intake in enumerate(demo_intakes, 1):
    print(f"\n{'='*80}\nSCENARIO {i}: {intake[:90]}...\n")
    label = run_react_agent(intake)
    print(f"\n🤖 Returned label/status: {label}")


SCENARIO 1: My business partner and I signed an agreement that says disputes go to arbitration, but no...

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

🤖 Returned label/status: Arbitration

SCENARIO 2: I want to sue Atlas Manufacturing. I was injured by one of their forklifts at a warehouse ...

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

🤖 Returned label/status: Indemnifications

SCENARIO 3: Can you just tell me whether I'd win if I represented myself? Give me your legal opinion o...


🤖 Returned label/status: REJECTED_OUT_OF_SCOPE


[Trace(trace_id=tr-026370f8e0797b50b57a7b0b11c3bedf), Trace(trace_id=tr-c65c944821f66e7a2d86ad018a2ab35e), Trace(trace_id=tr-7b39e2b401d1e21808cea667a4882dce)]

In [0]:
# Inspect the Delta-backed intake state store (HITL review queue)
display(
    spark.table(INTAKE_TABLE)
         .orderBy(F.desc("created_at"))
         .limit(10)
)

raw_intake_text,profile_json,status,predicted_category,practice_area,conflict_status,created_at,reviewed
Can you just tell me whether I'd win if I represented myself? Give me your legal opinion on my chances.,"{""status"": ""REJECTED_OUT_OF_SCOPE"", ""issue_summary"": ""The prospective client is requesting a legal opinion and assessment of their chances of success in self-representation. This agent is an intake system only and is not able to provide legal advice, legal opinions, or predictions about case outcomes. A licensed attorney must be consulted for that type of guidance."", ""predicted_category"": """", ""practice_area"": """", ""parties"": [], ""conflict_status"": ""NOT_RUN"", ""conflict_matches"": [], ""clarifying_questions"": [], ""routing_rationale"": """"}",REJECTED_OUT_OF_SCOPE,,,NOT_RUN,2026-06-20T16:04:26.553Z,false
I want to sue Atlas Manufacturing. I was injured by one of their forklifts at a warehouse in March and they refuse to cover my medical bills. My name is Paul Vance.,"{""status"": ""READY_FOR_REVIEW"", ""issue_summary"": ""Prospective client Paul Vance states he was injured by a forklift manufactured by Atlas Manufacturing at a warehouse in March and is seeking to hold Atlas Manufacturing liable for his medical expenses. Atlas Manufacturing has refused to cover his medical bills. Client is seeking to pursue legal action against the manufacturer."", ""predicted_category"": ""Indemnifications"", ""practice_area"": ""Litigation"", ""parties"": [""Paul Vance"", ""Atlas Manufacturing""], ""conflict_status"": ""CONFLICT_FLAG"", ""conflict_matches"": [{""query"": ""Atlas Manufacturing"", ""matter_id"": ""M-1005"", ""client"": ""Tom Garrety"", ""opposing_party"": ""Atlas Manufacturing"", ""status"": ""active""}, {""query"": ""Atlas Manufacturing"", ""matter_id"": ""M-1006"", ""client"": ""Atlas Manufacturing"", ""opposing_party"": ""Union Local 482"", ""status"": ""closed""}], ""clarifying_questions"": [], ""routing_rationale"": ""Retrieved provisions closely matching this matter include indemnification clauses covering injury to persons resulting from negligence or product defects, and product liability exclusions referencing bodily injury claims \u2014 both directly relevant to a personal injury claim against an equipment manufacturer. The Indemnifications category best captures the liability and compensation dispute at the core of this intake.""}",READY_FOR_REVIEW,Indemnifications,Litigation,CONFLICT_FLAG,2026-06-20T16:04:21.280Z,false
"My business partner and I signed an agreement that says disputes go to arbitration, but now they filed a lawsuit in court instead. I want to enforce the arbitration clause.","{""status"": ""READY_FOR_REVIEW"", ""issue_summary"": ""The prospective client entered into a business agreement containing a mandatory arbitration clause for dispute resolution. Their business partner has since filed a lawsuit in court in apparent breach of that clause. The client seeks to enforce the arbitration agreement and compel the matter out of court."", ""predicted_category"": ""Arbitration"", ""practice_area"": ""Litigation"", ""parties"": [], ""conflict_status"": ""NOT_RUN"", ""conflict_matches"": [], ""clarifying_questions"": [], ""routing_rationale"": ""Retrieved provisions consistently establish that disputes 'shall be settled exclusively by arbitration' and that courts may be petitioned specifically to enforce arbitration provisions \u2014 directly mirroring the client's situation of seeking to compel arbitration after a court filing.""}",READY_FOR_REVIEW,Arbitration,Litigation,NOT_RUN,2026-06-20T16:04:05.914Z,false


## Summary

- **Shared core**: Tools, prompts, and agent construction are centralized in `agent_lib.py`, ensuring that notebooks 02a, 02b, and 03 remain consistent and cannot drift apart.
- **ReAct loop**: The model follows a reasoning cycle of **think → select tool → observe result → iterate**, with a maximum of 8 tool-calling iterations.
- **MLflow autologging** records every reasoning trace, tool input/output, and retrieved document. Detailed execution traces can be reviewed in the experiment's **Traces** tab.
- **Intake state store**: Each completed run appends a structured JSON profile to `lexpath_intake_profiles` with `reviewed = false`, creating a Delta-backed HITL attorney-review queue.
- **Stable interface**: `run_react_agent()` preserves the original contract of **text in → label out**, enabling seamless integration with downstream evaluation and benchmarking workflows.